In [6]:
# ============================================
# 1. IMPORTS
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
import re
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================
# 2. LOAD DATASET
# ============================================
def load_lines(file):
    lines = {}
    with open(file, encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.split(" +++$+++ ")
            lines[parts[0]] = parts[-1].strip()
    return lines

def load_conversations(file):
    convs = []
    with open(file, encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.split(" +++$+++ ")
            ids = eval(parts[-1])
            convs.append(ids)
    return convs

def extract_pairs(lines, convs):
    pairs = []
    for conv in convs:
        for i in range(len(conv)-1):
            pairs.append([lines[conv[i]], lines[conv[i+1]]])
    return pairs

lines = load_lines("movie_lines.txt")
convs = load_conversations("movie_conversations.txt")
pairs = extract_pairs(lines, convs)

print("Total pairs:", len(pairs))

# ============================================
# 3. PREPROCESSING
# ============================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

pairs = [[clean_text(q), clean_text(a)] for q, a in pairs]

MAX_LEN = 8
pairs = [[q, a] for q, a in pairs if len(q.split()) < MAX_LEN and len(a.split()) < MAX_LEN]

# ✅ LIMIT TO 10K
pairs = pairs[:10000]

print("Final dataset size:", len(pairs))

# ============================================
# 4. VOCABULARY
# ============================================
word2index = {"<PAD>":0, "<SOS>":1, "<EOS>":2}
index2word = {0:"<PAD>", 1:"<SOS>", 2:"<EOS>"}
word_count = 3

def add_sentence(sentence):
    global word_count
    for word in sentence.split():
        if word not in word2index:
            word2index[word] = word_count
            index2word[word_count] = word
            word_count += 1

for q, a in pairs:
    add_sentence(q)
    add_sentence(a)

def sentence_to_tensor(sentence):
    indexes = [word2index[word] for word in sentence.split() if word in word2index]
    indexes.append(word2index["<EOS>"])
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

# ============================================
# 5. MODEL
# ============================================
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Parameter(torch.rand(hidden_size))

    def forward(self, hidden, encoder_outputs):
        energy = torch.tanh(self.attn(encoder_outputs))
        energy = torch.sum(self.v * energy, dim=2)
        weights = torch.softmax(energy, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs)
        return context

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.attention = Attention(hidden_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        embedded = self.embedding(x)
        context = self.attention(hidden[-1], encoder_outputs)
        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

# ============================================
# 6. INITIALIZATION
# ============================================
vocab_size = len(word2index)
embed_size = 128
hidden_size = 256

encoder = Encoder(vocab_size, embed_size, hidden_size).to(device)
decoder = Decoder(vocab_size, embed_size, hidden_size).to(device)

criterion = nn.CrossEntropyLoss()
enc_opt = optim.Adam(encoder.parameters(), lr=0.001)
dec_opt = optim.Adam(decoder.parameters(), lr=0.001)

# ============================================
# 7. TRAINING
# ============================================
def train(input_tensor, target_tensor):
    enc_opt.zero_grad()
    dec_opt.zero_grad()

    encoder_outputs, hidden, cell = encoder(input_tensor)
    decoder_input = torch.tensor([[word2index["<SOS>"]]], device=device)

    loss = 0

    for t in range(target_tensor.size(1)):
        output, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)
        loss += criterion(output, target_tensor[:, t])
        decoder_input = target_tensor[:, t].unsqueeze(1)

    loss.backward()
    enc_opt.step()
    dec_opt.step()

    return loss.item() / target_tensor.size(1)

# TRAIN LOOP
epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for q, a in pairs:
        input_tensor = sentence_to_tensor(q)
        target_tensor = sentence_to_tensor(a)

        loss = train(input_tensor, target_tensor)
        total_loss += loss

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# ============================================
# 8. SAVE MODEL (IMPORTANT)
# ============================================
torch.save(encoder.state_dict(), "encoder.pth")
torch.save(decoder.state_dict(), "decoder.pth")

# ============================================
# 9. CHATBOT
# ============================================
def evaluate(sentence):
    with torch.no_grad():
        input_tensor = sentence_to_tensor(clean_text(sentence))
        encoder_outputs, hidden, cell = encoder(input_tensor)

        decoder_input = torch.tensor([[word2index["<SOS>"]]], device=device)

        words = []
        for _ in range(10):
            output, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)
            topv, topi = output.topk(1)

            if topi.item() == word2index["<EOS>"]:
                break

            words.append(index2word.get(topi.item(), ""))
            decoder_input = topi.detach().view(1, 1)

        return " ".join(words)

# ============================================
# 10. RUN CHATBOT
# ============================================
while True:
    inp = input("You: ")
    if inp.lower() == "exit":
        break
    print("Bot:", evaluate(inp))

Total pairs: 221616
Final dataset size: 10000
Epoch 1, Loss: 51635.9064
Epoch 2, Loss: 42894.3388
Epoch 3, Loss: 36732.1175
Epoch 4, Loss: 31177.2985
Epoch 5, Loss: 26633.5291
Epoch 6, Loss: 22616.5710
Epoch 7, Loss: 19245.0145
Epoch 8, Loss: 16411.0448
Epoch 9, Loss: 14020.6379
Epoch 10, Loss: 12072.8315
You: hello
Bot: hey how ya doin
You: how are you?
Bot: i am afraid not
You: how are you
Bot: i am afraid not
You: hello
Bot: hey how ya doin
You: exit
